# Goal

Прогон исследования `23.3` против старой версии `18n_ppo_tr_frostbite_07` от 2026-09-27. 

Цель: получить сравнительную базу, чтобы понять, где сломалалась (?) логика. Сравниваем с `18n_study_23.3b`

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07_old.ipynb'

# GRID_SEARCH_SPACE

In [2]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [3]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = 75 # СОХРАНЕНИЕ РАНДОМ ПОСЕВА
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.vision_head.parent = dict(model='18d_world_model_09:40', weights='18d_world_model_09:40')
    HP.vision_head.is_trainable = False

    parent = '18n_ppo_tr_frostbite_07:124'
    
    HP.encoder.parent = dict(model='18d_world_model_09:40', weights=parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = parent 
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = None
    HP.test.env_ram_patches = None
    HP.test.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = [
        ['nine_lives'], 
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 512 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.vf_coef = 0.2
    HP.ppo.ent_coef = 'const(0.05)'
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.0 # НАМЕРЕННОЕ СОХРАНЕНИЕ ОШИБКИ
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP

# Results


Одни и те же входные данные, но разные результаты. Наблюдаются явно разные распределения на обоих марджин чартах.

Получается, что `18n_ppo_tr_frostbite_07_old.ipynb` и `18n_ppo_tr_frostbite_07.ipynb` действительно различаются в логике. 

Сравнил diff. Вспомнил, что было одно невинное изменение - использование `ob_features` с роллаута. В старое версии мы шли он сырых обзов и прогоняли их через `VisionHead` на стадии оптимизации. И там всё это было под `with torch.amp.autocast`. А в новой версии мы используем `ob_features` с роллаута, которые без AMP получены. Т.к. AMP приводит к снижению точности, то это скорее всего и объясняет разницу в результатах - агенты на старой версии из-за этой неточности получали что-ли какой-то бонус к exploration.

Тут быстро с Джемини обсудил, хотя он просто подтвердил мои соображения: https://share.google/aimode/G7EYw1CthodC1aIfX.

Можно попробовать по-разному: либо AMP на роллауте прикрутить, либо поиграться коэффициентом энтропии.

|Что|`18n_study_23.3b`|`18n_study_23.3c`|
|---|---|---|
|episode_r|<img src="./img/episode_r_23.3b.png">|<img src="./img/episode_r_23.3c.png">|
|episode_lp|<img src="./img/episode_lp_23.3b.png">|<img src="./img/episode_lp_23.3c.png">|

**Выводы**
1) попробовать пока через коэффициент энтропии - станет ли лучше?

# System

In [4]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [5]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07_old.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07_old',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_23.3c/18n_study_23.3c.ipynb',
 'optuna_study_name': '18n_study_23.3c',
 'optuna_study_serial': '23.3c',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_23.3c/18n_study_23.3c.optuna'}

In [6]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [7]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [8]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='test/levels_passed_mean',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [9]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [13]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 3
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-07 13:11:28,557] Using an existing study with name '18n_study_23.3c' instead of creating a new one.


2026.10.07-13:11:28.843972     0.258 >> Model instance registered, version=7
2026.10.07-13:11:28.856715     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch7.ipynb"
2026.10.07-13:11:28.857564     0.006 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07_old:7"; running launches=1
2026.10.07-13:12:00.363398     0.274 >> Model instance registered, version=8
2026.10.07-13:12:00.376751     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch8.ipynb"
2026.10.07-13:12:00.377238     0.006 >> 4.5 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07_old:8"; running launches=2
2026.10.07-13:12:31.473908     0.417 >> Model instance registered, version=9
2026.10.07-13:12:31.483310     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch9.ipynb"
2026.10.07-13:12:31.483691     0.005 >> 2.5 idle runners exist, submitted launch "18n_ppo_tr_frostbit

In [11]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-07 11:54:27,846] Using an existing study with name '18n_study_23.3c' instead of creating a new one.


2026.10.07-11:54:27.848146     5.029 >> Study statistics: 
2026.10.07-11:54:27.849003     0.001 >> 	Number of finished trials: 6
2026.10.07-11:54:27.849398     0.000 >> 	Number of pruned trials: 0
2026.10.07-11:54:27.849851     0.000 >> 	Number of complete trials: 6
2026.10.07-11:54:27.850483     0.001 >> Best trial:
2026.10.07-11:54:27.851147     0.001 >> 	Value: 3.1875
2026.10.07-11:54:27.851541     0.000 >> 	Model version: 1
2026.10.07-11:54:27.851904     0.000 >> 	Params: 


In [12]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.07-11:54:27.882058     0.030 >> Top trials:
2026.10.07-11:54:27.883053     0.001 >> 1) 18n_ppo_tr_frostbite_07_old:1, value=3.1875
2026.10.07-11:54:27.883716     0.001 >> 2) 18n_ppo_tr_frostbite_07_old:6, value=3.03125
2026.10.07-11:54:27.884446     0.001 >> 3) 18n_ppo_tr_frostbite_07_old:2, value=2.90625
